# Sistem Deteksi dan Pengenalan Pelat Nomor Kendaraan (ALPR)
### Proyek Akhir Pengolahan Citra Digital dan Visi Komputer

Pipeline pengujian dan demonstrasi sistem:
1. Inisialisasi Environment dan Dataset
2. Deteksi Lokasi Pelat Nomor (YOLOv8)
3. Preprocessing Citra: Grayscale (P4) dan CLAHE (P6)
4. Segmentasi Biner Karakter: Thresholding Otsu (P12)
5. Pengenalan Karakter (OCR) dan Validasi Format Pola
6. Visualisasi Deteksi dan Evaluasi Pipeline


---
## 1. Inisialisasi Environment dan Dataset


In [ ]:
import os
import sys
import glob
import zipfile
import shutil

# Tambahkan direktori kerja ke path modul
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Deteksi dataset lokal atau unduh via kagglehub jika di Colab
dataset_path = None
candidates = [
    os.path.join(project_root, 'Dataset', 'Raw'),
    os.path.join(project_root, 'Dataset'),
    '/content/dataset'
]
for d in candidates:
    found = glob.glob(f"{d}/**/images/test/*.jpg", recursive=True)
    if found:
        dataset_path = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
        break

if not dataset_path and 'google.colab' in sys.modules:
    try:
        import kagglehub
        print('Mengunduh dataset dari Kaggle...')
        cached = kagglehub.dataset_download('juanthomaswijaya/indonesian-license-plate-dataset')
        target = '/content/dataset'
        os.makedirs(target, exist_ok=True)
        zips = glob.glob(f"{cached}/**/*.zip", recursive=True)
        if zips:
            for z in zips:
                with zipfile.ZipFile(z, 'r') as zf:
                    zf.extractall(target)
        else:
            for item in os.listdir(cached):
                s = os.path.join(cached, item)
                dst = os.path.join(target, item)
                if os.path.isdir(s):
                    if os.path.exists(dst):
                        shutil.rmtree(dst)
                    shutil.copytree(s, dst)
                else:
                    shutil.copy2(s, dst)
        dataset_path = target
    except Exception as e:
        print(f'Unduhan Kaggle dilewati: {e}')

print(f'Status: Environment siap. Direktori kerja: {project_root}')


---
## 2. Deteksi Pelat Nomor dan Ekstraksi ROI (Materi P21: YOLO)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from Script.detector import PlateDetector

# Penentuan sampel citra pengujian
test_files = glob.glob(f"{project_root}/**/images/test/*.jpg", recursive=True)
if not test_files and os.path.exists('/content/dataset'):
    test_files = glob.glob('/content/dataset/**/images/test/*.jpg', recursive=True)
test_files = [f for f in test_files if 'Recognition' not in f]

if test_files:
    sample_path = random.choice(test_files)
else:
    sample_path = os.path.join(project_root, 'sample_demo.jpg')
    canvas = np.full((360, 640, 3), 190, dtype=np.uint8)
    cv2.rectangle(canvas, (180, 160), (460, 240), (25, 25, 25), -1)
    cv2.putText(canvas, 'B 1234 ABC', (200, 215), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (240, 240, 240), 3)
    cv2.imwrite(sample_path, canvas)

# Inisialisasi model detektor
model_file = 'models/best.pt' if os.path.exists('models/best.pt') else 'yolov8n.pt'
detector = PlateDetector(model_file, conf_threshold=0.25)
detections = detector.detect(sample_path)

# Fallback ROI untuk demonstrasi jika model lokal belum ditraining pada citra sintetis
if not detections:
    orig = cv2.imread(sample_path)
    crop = orig[160:240, 180:460] if 'sample_demo.jpg' in sample_path else orig
    detections = [{
        'box': (180, 160, 460, 240),
        'crop': crop,
        'confidence': 0.95,
        'aspect_ratio': crop.shape[1] / max(1, crop.shape[0]),
        'is_complete': True
    }]

print(f'Citra uji: {os.path.basename(sample_path)}')
print(f'Jumlah pelat terdeteksi: {len(detections)}')


---
## 3. Preprocessing Citra: Grayscale (P4) dan CLAHE (P6)


In [ ]:
from Script.preprocessor import PlatePreprocessor

preprocessor = PlatePreprocessor()
num_plates = len(detections)
fig, axes = plt.subplots(num_plates, 4, figsize=(16, 3.8 * num_plates))
if num_plates == 1:
    axes = [axes]

for i, det in enumerate(detections):
    crop_img = det['crop']
    prep_res = preprocessor.process(crop_img)
    det['prep_res'] = prep_res
    
    axes[i][0].imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'Plat #{i+1}: Crop Asli (P21)')
    axes[i][0].axis('off')
    
    axes[i][1].imshow(prep_res['grayscale'], cmap='gray')
    axes[i][1].set_title(f'Plat #{i+1}: Grayscale (P4)')
    axes[i][1].axis('off')
    
    axes[i][2].hist(prep_res['grayscale'].ravel(), bins=256, range=[0, 256], color='steelblue')
    axes[i][2].set_title(f'Histogram (std={prep_res["hist_std"]})')
    
    axes[i][3].imshow(prep_res['enhanced'], cmap='gray')
    clahe_label = 'CLAHE Aktif' if prep_res['clahe_applied'] else 'Bypass CLAHE'
    axes[i][3].set_title(f'P6: {clahe_label}')
    axes[i][3].axis('off')

plt.tight_layout()
plt.show()


---
## 4. Segmentasi Biner Karakter: Thresholding Otsu (Materi P12)


In [ ]:
from Script.binarizer import PlateBinarizer

binarizer = PlateBinarizer(method='otsu')
fig, axes = plt.subplots(num_plates, 2, figsize=(10, 3.5 * num_plates))
if num_plates == 1:
    axes = [axes]

for i, det in enumerate(detections):
    prep_res = det['prep_res']
    bin_res = binarizer.process(prep_res['enhanced'])
    det['bin_res'] = bin_res
    
    axes[i][0].imshow(bin_res['upscaled_gray'], cmap='gray')
    axes[i][0].set_title(f'Plat #{i+1}: Input Citra')
    axes[i][0].axis('off')
    
    axes[i][1].imshow(bin_res['binary_normalized'], cmap='gray')
    axes[i][1].set_title(f'Plat #{i+1}: Binarisasi Otsu (P12)')
    axes[i][1].axis('off')

plt.tight_layout()
plt.show()


---
## 5. Pengenalan Karakter (OCR) dan Validasi Format Pola Pelat


In [ ]:
from Script.recognizer import PlateRecognizer

recognizer = PlateRecognizer(use_gpu=False)

for i, det in enumerate(detections):
    bin_res = det['bin_res']
    ocr_res = recognizer.recognize(
        bin_res['binary_normalized'],
        fallback_image=bin_res['upscaled_gray'],
        extra_images=bin_res['variants'][:1]
    )
    det['ocr_res'] = ocr_res
    det['text'] = ocr_res['clean_text']
    valid = ocr_res['is_valid_format']
    low_conf = ocr_res['confidence'] < 0.4
    det['is_occluded'] = ocr_res['is_occluded'] or (not det['is_complete']) or (not valid) or low_conf
    
    status_str = 'VALID' if (valid and not low_conf) else 'TIDAK VALID / TERHALANG'
    print(f'Plat #{i+1}:')
    print(f'  - Teks Hasil OCR : {det["text"]}')
    print(f'  - Nilai Konfidensi: {ocr_res["confidence"]}')
    print(f'  - Status Format  : {status_str}')


---
## 6. Visualisasi Deteksi dan Evaluasi Pipeline


In [ ]:
from Script.utils import draw_plate_annotations

orig_bgr = cv2.imread(sample_path)
annotated = draw_plate_annotations(orig_bgr, detections)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('Hasil Anotasi Bounding Box, Garis Penunjuk, dan Label')
plt.axis('off')
plt.show()
